#  **Quality Control: Defect Detection in Manufacturing Images**

## **Problem Statement**

In modern manufacturing industries, maintaining product quality is critical to ensure reliability, safety, and customer satisfaction. Traditional quality inspection methods rely heavily on manual visual inspection, which is time-consuming, inconsistent, and prone to human error.

The objective of this project is to develop an automated defect detection system using machine learning techniques. The system analyzes surface images of manufactured materials and classifies them into different defect categories such as Crazing, Inclusion, Patches, Pitted Surface, Rolled-in Scale, and Scratches.

Using the NEU Surface Defect Database, this project applies image preprocessing, feature engineering, and multiple machine learning models to accurately identify defects. The final system is deployed as an interactive web application, allowing users to upload images and receive real-time predictions.

This solution aims to improve efficiency, reduce inspection time, and enhance the overall quality control process in manufacturing environments.

## **Real-World Motivation**

In manufacturing industries such as steel production, automotive, and electronics, surface defects can significantly impact product quality and safety. Even small defects may lead to product failure, increased maintenance costs, and customer dissatisfaction.

Manual inspection is widely used but has several limitations. It is slow, inconsistent, and depends heavily on human expertise, which may vary over time. As production scales increase, manual inspection becomes inefficient and costly.

Therefore, there is a strong need for an automated system that can quickly and accurately detect defects in manufacturing images.

## **Why Automation is Needed**

Automation in defect detection offers several advantages:

- **Speed:** Machines can analyze images much faster than humans.
- **Consistency:** Eliminates human errors and fatigue-related mistakes.
- **Scalability:** Can handle large volumes of production data efficiently.
- **Cost Reduction:** Reduces dependency on manual labor.
- **Real-Time Monitoring:** Enables immediate detection and response.

By leveraging machine learning, the system can learn patterns from data and improve detection accuracy over time, making it highly suitable for modern industrial applications.

# **EDA (Exploratory Data Analysis)**

## Import Libraries

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Load dataset

In [17]:
import pandas as pd

df = pd.read_csv(r"C:\defect_detection\dataset.csv")

df.head()

,0,1,2,3,4,5,6,7,8,9,...,4087,4088,4089,4090,4091,4092,4093,4094,4095,4096
0,175,196,202,155,195,180,160,168,208,165,...,137,129,145,143,142,175,159,142,137,crazing
1,169,192,205,216,177,179,217,190,200,218,...,89,110,105,108,96,84,86,87,71,crazing
2,102,95,96,108,132,118,137,135,116,109,...,114,93,92,106,105,102,106,92,85,crazing
3,152,174,191,158,141,176,150,171,148,145,...,189,208,164,149,164,166,164,162,135,crazing
4,167,184,126,113,159,140,157,128,146,154,...,59,59,53,51,49,40,49,25,48,crazing


In [18]:
df.shape

(1800, 4097)

In [19]:
df.iloc[:, -1].value_counts()

4096
crazing            300
inclusion          300
patches            300
pitted_surface     300
rolled-in_scale    300
scratches          300
Name: count, dtype: int64

### Class distribution

In [20]:
import matplotlib.pyplot as plt

df.iloc[:, -1].value_counts().plot(kind='bar')
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.savefig("../outputs/plots/class_distribution.png")
plt.close()
plt.show()

# **Preprocessing**

In [21]:
df.isnull().sum()

0       0
1       0
2       0
3       0
4       0
       ..
4092    0
4093    0
4094    0
4095    0
4096    0
Length: 4097, dtype: int64

In [22]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

### Normalize data
This improves model performance

In [23]:
X = X / 255.0

### Histogram

In [24]:
import matplotlib.pyplot as plt

# Mean pixel value per image
pixel_mean = X.iloc[:, :4096].mean(axis=1)

plt.figure()
plt.hist(pixel_mean, bins=30)
plt.title("Pixel Intensity Distribution")
plt.xlabel("Mean Pixel Value")
plt.ylabel("Frequency")

plt.savefig("../outputs/plots/histogram.png")
plt.close()

### Boxplot

In [25]:
plt.figure()

# Take only a few features (important!)
plt.boxplot(X.iloc[:, :5])

plt.title("Boxplot of First 5 Features")
plt.xticks(rotation=90)
plt.savefig("../outputs/plots/boxplot.png")
plt.close()

### Correlation Matrix

In [26]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))

# Using only first 50 columns (important for performance)
sns.heatmap(X.iloc[:, :50].corr(), cmap='coolwarm')

plt.title("Correlation Matrix")
plt.savefig("../outputs/plots/correlation_matrix.png")
plt.close()

# **Feature Engineering**

In [12]:
mean_pixel = X.mean(axis=1)
std_pixel = X.std(axis=1)

import pandas as pd
X = pd.concat([X, mean_pixel.rename('mean_pixel'), std_pixel.rename('std_pixel')], axis=1)

In [13]:
import cv2
import numpy as np

def extract_edges(row):
    pixels = row.iloc[:4096]   # ALWAYS take first 4096 values
    img = np.array(pixels).reshape(64, 64)
    
    edges = cv2.Canny(img.astype('uint8'), 100, 200)
    return edges.mean()

X['edge_feature'] = X.apply(extract_edges, axis=1)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_8976\1648910712.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['edge_feature'] = X.apply(extract_edges, axis=1)


## Save processed dataset

In [14]:
processed_df = X.copy()
processed_df['label'] = y

processed_df.to_csv("../data/processed_dataset.csv", index=False)

print("✅ Processed dataset saved!")

✅ Processed dataset saved!
